In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.5 Eigenvalues, Diagonalization, and the SVD

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Mathematical & Computational Foundations",
    number="0.5",
    title="Eigenvalues, Diagonalization, and the SVD",
    blurb="The spectral theorem, the generalized eigenproblem behind every "
    "normal mode, and the singular value decomposition — the master "
    "factorization that powers least squares, data compression, and "
    "the tensor networks of many-body quantum physics.",
    difficulty="advanced",
    estimate="100–130 min",
)

## Notebook overview

Eigenvalues we already know as the invariant directions of a linear map; what
we have probably *not* seen is how a computer finds them, and that it does so
without ever touching the characteristic polynomial, whose roots are far too
ill-conditioned to trust (a direct callback to the conditioning lessons of
[§0.1](floating-point.ipynb) and [§0.4](linear-systems.ipynb)). This notebook
builds the numerical eigenproblem from the well-behaved
symmetric case and the **spectral theorem**, through the **QR algorithm** that
`scipy.linalg.eigh` actually runs, to the **generalized eigenproblem** $K\mathbf
v=\lambda M\mathbf v$ that is the literal engine of every normal-mode calculation
in [§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb),
[§2.6](../02-classical-mechanics/rigid-body.ipynb), and
[§2.7](../02-classical-mechanics/small-oscillations.ipynb).

Its second half is the **singular value decomposition**: the one factorization
that works for *any* matrix, square or not, and arguably the most useful in all
of applied mathematics. We will see its geometry (every linear map is a rotation,
a scaling, and another rotation), prove the **Eckart–Young** theorem (the best
low-rank approximation is the truncated SVD), and watch it strip noise from a
nearly-low-rank matrix. That last move (keep the few large singular values,
discard the small tail) is, almost verbatim, how the **tensor networks** of
many-body quantum physics tame an exponentially large Hilbert space, a pointer
we plant for a later volume.

Throughout we lean on [§0.4](linear-systems.ipynb): its condition number $\kappa$, its QR
factorization, and its "never invert" rule all return here. The only animation is
the SVD's geometric action, where motion genuinely shows what a still cannot; the
spectra and error plots are static.

> **How to read the checks.** Each exercise ends with a `validate` call against
> an independent fact: a reconstruction $A=V\Lambda V^\top$, a known normal-mode
> frequency, an Eckart–Young error equal to $\sigma_{k+1}$. A ✓ is strong
> evidence; a ✗ is a prompt to *locate the discrepancy*, not a verdict.

> **Scope.** A working review, not a course in matrix computations. See Trefethen
> & Bau, *Numerical Linear Algebra* {cite}`trefethen1997` (the eig and SVD
> lectures) and Golub & Van Loan {cite}`golubvanloan`.

## Theory in brief

### The eigenproblem, and why not the characteristic polynomial

An eigenpair of a square matrix $A$ satisfies

```{math}
:label: eq-eigprob
A\mathbf v = \lambda\mathbf v,
```

so $\mathbf v$ is a direction the map only stretches, by the factor $\lambda$.
We learned to find $\lambda$ as the roots of $\det(A-\lambda I)=0$, but that is
precisely what a numerical eigensolver refuses to do: the map from a polynomial's
coefficients to its roots is wildly ill-conditioned (Wilkinson's classic warning),
so forming the characteristic polynomial and rooting it loses most of the digits.
Instead, eigenvalues are found by *iteration*: repeated orthogonal
transformations that drive the matrix toward triangular form.

### The symmetric case and the spectral theorem

The friendliest (and most physical) case is a real symmetric $A=A^\top$
(observables, quadratic forms, mass and stiffness matrices). Its eigenvalues are
real and its eigenvectors can be chosen orthonormal, so $A$ diagonalizes as

```{math}
:label: eq-spectral
A = V\Lambda V^\top = \sum_i \lambda_i\,\mathbf v_i\mathbf v_i^\top,
\qquad V^\top V = I,
```

the **spectral theorem**: $A$ is a weighted sum of orthogonal projectors onto its
eigendirections. `scipy.linalg.eigh` returns exactly this $V$ and $\Lambda$.

### The QR algorithm

How does `eigh` get there? By the **QR algorithm**, which reuses the QR
factorization of [§0.4](linear-systems.ipynb) in a strikingly simple loop:
factor, then multiply the factors back in the opposite order,

```{math}
:label: eq-qralg
A_k = Q_k R_k, \qquad A_{k+1} = R_k Q_k,
```

and repeat. Each $A_{k+1}=Q_k^\top A_k Q_k$ is an orthogonal similarity of $A$, so
the eigenvalues are preserved, and the iterates converge to (block-)triangular
form with the eigenvalues marching out onto the diagonal. (Why so simple a loop
converges is genuinely subtle; Trefethen & Bau, *Numerical Linear Algebra*,
Lectures 28–29, supply the convergence theory.)

### The generalized eigenproblem

Normal modes do not come as $A\mathbf v=\lambda\mathbf v$ but as

```{math}
:label: eq-genev
K\mathbf v = \lambda M\mathbf v,
```

with a stiffness matrix $K$ and a mass matrix $M\ne I$: exactly the form derived
in [§2.7](../02-classical-mechanics/small-oscillations.ipynb). It is solved by
`eigh(K, M)`, whose eigenvectors are $M$-orthonormal; the
$\lambda$ are the squared normal-mode frequencies. This is the engine under
[§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb),
[§2.6](../02-classical-mechanics/rigid-body.ipynb), and
[§2.7](../02-classical-mechanics/small-oscillations.ipynb).

### The singular value decomposition

Eigenvalues need a square matrix; the **SVD** needs nothing of the sort. Every
$m\times n$ matrix factors as

```{math}
:label: eq-svd
A = U\Sigma V^\top, \qquad U^\top U = I,\ V^\top V = I,
```

with $\Sigma$ diagonal holding the **singular values** $\sigma_1\ge\sigma_2\ge
\cdots\ge 0$. Geometrically every linear map is therefore the same three acts:
rotate ($V^\top$), scale along axes ($\Sigma$), rotate ($U$), so a matrix sends
the unit sphere to an ellipsoid whose semi-axes are the $\sigma_i$. The singular
values are $\sigma_i=\sqrt{\lambda_i(A^\top A)}$, tying the SVD back to the
symmetric eigenproblem. (That every matrix admits this factorization is a
theorem, not an observation; Trefethen & Bau, *Numerical Linear Algebra*,
Lectures 4–5, prove it.)

### Low-rank approximation: Eckart–Young

Truncating the SVD to its $k$ largest singular values gives the provably best
rank-$k$ approximation of $A$, and the error is exactly the first discarded
singular value:

```{math}
:label: eq-eckart
\min_{\operatorname{rank}B\le k}\lVert A-B\rVert_2 = \sigma_{k+1},
\qquad B = \sum_{i=1}^{k}\sigma_i\,\mathbf u_i\mathbf v_i^\top.
```

This single fact is the mathematical core of data compression and of the
tensor-network methods of many-body quantum physics alike. (Trefethen & Bau,
*Numerical Linear Algebra*, Lecture 5, gives the short proof.)

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.linalg import eigh, svd, qr, eigvalsh

from ecp import validate
from ecp.animate import show

np.set_printoptions(precision=4, suppress=True)

## Exercise 1 — The symmetric eigenproblem and the spectral theorem

The cleanest entry point is a real symmetric matrix, where {eq}`eq-spectral`
promises real eigenvalues and orthonormal eigenvectors. Take the explicit matrix

$$
A = \begin{bmatrix} 2 & 1 & 0 \\ 1 & 3 & 1 \\ 0 & 1 & 2 \end{bmatrix},
$$

which is symmetric, so the spectral theorem applies.

1. Diagonalize it with `scipy.linalg.eigh`.
2. Verify both halves of {eq}`eq-spectral`: the eigenvectors are orthonormal
   ($V^\top V=I$) and $A$ is rebuilt as $V\Lambda V^\top$.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(V1.T @ V1, np.eye(3), "eigenvectors are orthonormal (VᵀV=I)", atol=1e-10)
validate.close(V1 @ np.diag(w1) @ V1.T, A1, "A = VΛVᵀ (spectral theorem)", atol=1e-10)

## Exercise 2 — Why not the characteristic polynomial

It is worth seeing *why* numerical eigensolvers avoid the textbook route. For the
same matrix $A=\begin{bmatrix}2&1&0\\1&3&1\\0&1&2\end{bmatrix}$ of Exercise 1, we
compare two routes against an exact reference. Because the coefficients-to-roots
map is ill-conditioned (the same kind of amplification $\kappa$ measured in
[§0.4](linear-systems.ipynb)), the polynomial route loses accuracy even on this
benign $3\times3$.

1. Compute the exact eigenvalues symbolically (`sympy.Matrix.eigenvals`) as the
   reference.
2. Compute them numerically with `scipy.linalg.eigvalsh` (backward-stable).
3. Form the characteristic polynomial's coefficients (`numpy.poly`) and root them
   (`numpy.roots`); compare both errors against the reference.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    err_eigh < err_charpoly,
    "the QR-based eigensolver beats rooting the characteristic polynomial",
    f"eigh {err_eigh:.2e} < charpoly {err_charpoly:.2e}",
)

## Exercise 3 — The QR algorithm from scratch

To see how `eigh` actually converges, we implement the unshifted QR algorithm of
{eq}`eq-qralg` using the very QR factorization built in
[§0.4](linear-systems.ipynb). Take the explicit
symmetric matrix

$$
A = \begin{bmatrix} 4 & 1 & 0 \\ 1 & 3 & 1 \\ 0 & 1 & 2 \end{bmatrix}.
$$

1. Iterate $A_k=Q_kR_k$, $A_{k+1}=R_kQ_k$ with `scipy.linalg.qr`; each step is an
   orthogonal similarity, so the spectrum is preserved.
2. Track the off-diagonal norm decaying as the diagonal converges to the
   eigenvalues ({numref}`fig-eigsvd-qr-conv`).
3. Confirm the converged diagonal matches `scipy.linalg.eigvalsh`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    diag_final,
    np.sort(eigvalsh(A3)),
    "the QR algorithm converges to the eigenvalues",
    atol=1e-6,
)

## Exercise 4 — The generalized eigenproblem (callback to 2.7)

Normal modes arrive as $K\mathbf v=\lambda M\mathbf v$, {eq}`eq-genev`, with a
mass matrix that is not the identity. Take the small-angle double pendulum of
[§2.7](../02-classical-mechanics/small-oscillations.ipynb) with equal bobs and
rods ($m=\ell=1$, $g=9.81$), whose mass and
stiffness matrices are

$$
M = m\ell^2\begin{bmatrix} 2 & 1 \\ 1 & 1 \end{bmatrix}, \qquad
K = mg\ell\begin{bmatrix} 2 & 0 \\ 0 & 1 \end{bmatrix}.
$$

1. Solve the generalized problem with `scipy.linalg.eigh(K, M)`; the eigenvalues
   are the squared normal-mode frequencies.
2. Confirm they reproduce the analytic doubles
   $\omega_\pm^2=(2\mp\sqrt2)\,g/\ell$ — the same modes obtained by direct
   integration in [§1.3](../01-elementary-mechanics/double-pendulum.ipynb) and
   by linearization in
   [§2.7](../02-classical-mechanics/small-oscillations.ipynb), here from one
   eigensolve.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    omega,
    omega_analytic,
    "generalized eig gives the double-pendulum modes √((2∓√2)g/ℓ)",
    rtol=1e-9,
)

## Exercise 5 — The SVD: definition and geometry (worked animation)

The SVD, {eq}`eq-svd`, applies to any matrix, and its content is geometric. We
take two explicit matrices. First, to check the algebra, the $3\times2$

$$
A = \begin{bmatrix} 3 & 1 \\ 1 & 3 \\ 2 & 2 \end{bmatrix}:
$$

1. Compute $U,\Sigma,V^\top$ with `scipy.linalg.svd`, verify the reconstruction
   $A=U\Sigma V^\top$, and confirm $\sigma_i=\sqrt{\lambda_i(A^\top A)}$
   (`scipy.linalg.eigvalsh` on $A^\top A$).

Then, to *see* the geometry, take the $2\times2$

$$
B = \begin{bmatrix} 3 & 1 \\ 0 & 2 \end{bmatrix},
$$

which sends the unit circle to an ellipse.

2. Animate the map decomposed into its three SVD acts ($V^\top$ rotates,
   $\Sigma$ stretches along the axes, $U$ rotates) with `FuncAnimation`.
3. Confirm the resulting ellipse has semi-axes exactly $\sigma_1,\sigma_2$
   ({numref}`fig-eigsvd-ellipse`) — the claim the validation checks against the
   animated points.

In [ ]:
# (solution hidden on the public site)


### Validation 5a — the SVD algebra

In [ ]:
validate.close(U5 @ np.diag(s5) @ Vt5, A5, "A = UΣVᵀ reconstructs A", atol=1e-10)
validate.close(s5, sigma_from_AtA, "σ = √eig(AᵀA)", atol=1e-10)

Now the geometry of $B=\begin{bmatrix}3&1\\0&2\end{bmatrix}$. Fix the SVD sign
freedom so both $U$ and $V^\top$ are proper rotations, then animate the unit
circle through the three acts.

In [ ]:
# (solution hidden on the public site)


### Validation 5b — the geometry of the data

In [ ]:
validate.close(
    np.sort(ellipse_semi_axes)[::-1],
    np.sort(sb)[::-1],
    "the unit circle maps to an ellipse with semi-axes σᵢ",
    rtol=1e-3,
)
# (rtol reflects measuring the semi-axes as the max/min radius over a finite
# sampling of the mapped ellipse: a discretization, not a physics, tolerance.)

## Exercise 6 — Low-rank approximation and Eckart–Young

Here is the centrepiece. Eckart–Young, {eq}`eq-eckart`, says the truncated SVD is
the *best* rank-$k$ approximation and that its spectral-norm error is exactly the
first singular value we discarded, $\sigma_{k+1}$. We test this on a fixed,
reproducible matrix: **the $8\times6$ matrix $B$ whose entries are
`numpy.random.default_rng(0).standard_normal((8, 6))`** (seed 0, shape $8\times6$,
stated so the result is unambiguous).

1. For $k=1,2,3,4$ form the rank-$k$ truncation
   $\sum_{i\le k}\sigma_i\mathbf u_i\mathbf v_i^\top$ from the
   `scipy.linalg.svd` factors.
2. Confirm its spectral-norm error (`numpy.linalg.norm(..., 2)`) equals
   $\sigma_{k+1}$ ({numref}`fig-eigsvd-eckart`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    errors_k,
    sigma_next,
    "the rank-k truncated-SVD error equals σ_(k+1) (Eckart–Young)",
    rtol=1e-6,
)

## Exercise 7 — Compression: a low-rank matrix in noise (student exercise)

Real data is rarely exactly low-rank, but it is often *nearly* so, and the SVD
exposes that at a glance. Build the explicit matrix

$$
M_{ij} = \sin(2\pi x_i) + x_i\,g_j + 0.01\,\eta_{ij},
$$

with $x=\texttt{linspace}(0,1,40)$, $g=\texttt{linspace}(1,2,30)$, and noise
$\eta=\texttt{default\_rng(0).standard\_normal((40,30))}$ (seed 0, shape
$40\times30$). The first two terms are each a rank-1 outer product, so $M$ is
**rank 2 plus a small noise floor**.

1. Compute the singular values with `scipy.linalg.svd` and see two dominate
   before a cliff.
2. Reconstruct the rank-2 matrix from the truncated factors (broadcast the top
   two singular values into the product) and check the noise is gone.
3. Quantify $\sigma_3/\sigma_1$ as the effective-rank signal.

There is no animation here: this is a spectrum-and-reconstruction analysis, not
motion; a ✗ points at the construction of $M$ or the truncation.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    s7[2] / s7[0] < 0.05,
    "the matrix is effectively rank 2: the noise lives in the tiny tail of the spectrum",
    f"σ₃/σ₁ = {s7[2] / s7[0]:.4f}",
)

## Exercise 8 — The SVD → tensor networks (synthesis and a forward pointer)

The move just made in Exercise 7 (keep the few large singular values, discard
the small tail) is, almost word for word, how the **tensor networks** of
many-body quantum physics are built. A quantum state of $N$ particles lives in a
Hilbert space of dimension $2^N$, hopelessly large; but a *physical* state
(a ground state of a local Hamiltonian) has, across any bipartition, an
**entanglement spectrum** (the singular values of the state reshaped into a
matrix) that decays rapidly, just like the noisy matrix above. A
**matrix-product state (MPS)** exploits exactly this: truncate each bond's SVD to
the largest few singular values (the "bond dimension"), and the $2^N$ cost
collapses to something linear in $N$. **DMRG** is the algorithm that does this
truncation variationally to find ground states.

We will not build an MPS here; that is the subject of a later many-body/quantum
volume. As a one-line synthesis, we verify the principle that makes it work:
keeping just the top two singular triples of the Exercise-7 matrix already
captures essentially all of its Frobenius norm: the same statement, for our toy
matrix, as "a low-entanglement state is well approximated by a small bond
dimension."

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    captured > 0.99,
    "keeping a few singular values captures almost all of the matrix — the "
    "principle behind tensor-network (MPS/DMRG) compression",
    f"captured fraction = {captured:.4f}",
)

## Notebook summary

- The symmetric eigenproblem and the **spectral theorem** ($A=V\Lambda V^\top$, $V^\top V=I$),
  why the characteristic polynomial is the wrong tool, and the **QR algorithm** built from
  scratch; the generalized eigenproblem (callback to
  [§2.7](../02-classical-mechanics/small-oscillations.ipynb)).
- The **SVD** ($A=U\Sigma V^\top$, $\sigma=\sqrt{\mathrm{eig}(A^\top A)}$), low-rank
  approximation and the Eckart–Young theorem, compression in noise, and the SVD as the seed of
  tensor-network methods.

## Outlook

- **Non-symmetric eigenproblems.** Complex spectra, non-orthogonal eigenvectors,
  and defective matrices with a Jordan form: numerically delicate, and a reason
  the symmetric case is so prized.
- **Power and inverse iteration.** The cheapest way to one extreme eigenpair;
  the seed of PageRank and of ground-state methods in §5.
- **PCA is the SVD of a data matrix**: a forward link to the least-squares
  fitting of [§0.8](fitting-least-squares.ipynb), where the SVD also gives the
  pseudoinverse for rank-deficient problems.
- **Tensor networks (MPS/DMRG)** take the Eckart–Young truncation to many-body
  quantum states, in a later volume.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()